In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import hashlib
from langchain_experimental.text_splitter import SemanticChunker
from pydantic import BaseModel
from langchain_openai import OpenAIEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI
from typing import Optional

from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader
)

C:\Users\mylav\AppData\Local\Temp\ipykernel_10476\4004835581.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
d:\nuevosol\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [4]:
loader = PyMuPDFLoader("StructuralReport.pdf")
docs = loader.load()
embeddings = OpenAIEmbeddings()
chunker = SemanticChunker(embeddings)
chunks = chunker.split_documents(docs)

In [5]:
len(chunks)

27

In [6]:
model = ChatGroq(model="qwen/qwen3-32b")
#model = ChatOpenAI(model="gpt-4o")

In [7]:
def get_text_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

In [8]:
vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

C:\Users\mylav\AppData\Local\Temp\ipykernel_10476\4169331872.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)


In [9]:
def add_chunks(chunks):
    for chunk in chunks:
        chunk_hash = get_text_hash(chunk.page_content)
        # Check if hash already exists in metadata
        existing = vectordb.get(where={"chunk_hash": chunk_hash})
        if not existing["ids"]:  # only add if not found
            vectordb.add_texts(
                texts=[chunk.page_content],
                metadatas=[{"chunk_hash": chunk_hash, **chunk.metadata}]
            )

In [10]:
add_chunks(chunks=chunks)

In [11]:
retriever = vectordb.as_retriever(
    search_kwargs={
        "k": 5,
    }
)

In [12]:
prompt = """
You are a professional with expertise in Solar Energy and Construction Site Engineering. 
Answer the user query : {query} based on the context provided below
Context:{context}
Carefully analyze the provided context and extract the following parameters asked by the user:

• Ground snow load : the weight of snow expected on the ground surface, used for site load calculations.  
• Roof snow load : the snow weight acting directly on building roofs, critical for roof design safety.  
• Wind speed : the maximum expected wind velocity at the site, used to calculate wind pressures on structures.  
• Seismic parameters (SS, S1, SDS, SD1) : values representing site-specific seismic accelerations for short and long periods, essential for earthquake-resistant design.

Instructions:
- Return the exact numerical value.    
- If multiple values are mentioned, choose the one clearly marked as applicable to the project site.  
- If a parameter is missing, do not create your value, return -1.0 as value.  
- Preserve the original units (psf, kN/m², mph, m/s, g, etc.) without conversion.  
"""
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("user","{prompt}")
    ]
)

In [20]:
query = """Provide me the parameters 
• Ground snow load
• Roof snow load
• Wind speed
• Seismic parameters (S_{S}, S_{1}, S_{DS}, S_{D1})
"""

In [21]:
query

'Provide me the parameters \n• Ground snow load\n• Roof snow load\n• Wind speed\n• Seismic parameters (S_{S}, S_{1}, S_{DS}, S_{D1})\n'

In [14]:
class Seismic_parameters(BaseModel):
    SS:float
    S1:float
    SDS:float
    SD1:float

class Site_Attributes(BaseModel):
    ground_snow_load: float
    roof_snow_load: float
    wind_Speed: float
    seismic_parameters: Seismic_parameters

In [15]:
model_with_structure = model.with_structured_output(Site_Attributes)

In [16]:
chain = retriever | prompt_template | model_with_structure

In [17]:
response = chain.invoke(query)

In [18]:
response

Site_Attributes(ground_snow_load=60.0, roof_snow_load=42.0, wind_Speed=115.0, seismic_parameters=Seismic_parameters(SS=0.303, S1=0.109, SDS=0.303, SD1=0.109))

In [22]:
json_str=response.model_dump_json()
json_str

'{"ground_snow_load":60.0,"roof_snow_load":42.0,"wind_Speed":115.0,"seismic_parameters":{"SS":0.303,"S1":0.109,"SDS":0.303,"SD1":0.109}}'

In [18]:
## SS and S1 are not extracted properly

In [117]:
docs = retriever.invoke(query)

In [118]:
docs

[Document(metadata={'creationDate': "D:20260325192527+05'30'", 'subject': '', 'source': 'StructuralReport.pdf', 'title': 'March 3rd, 2012', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-25T19:25:27+05:30', 'producer': 'Microsoft® Word 2021', 'moddate': '2026-03-25T19:25:27+05:30', 'author': 'ruly', 'trapped': '', 'format': 'PDF 1.7', 'file_path': 'StructuralReport.pdf', 'keywords': '', 'page': 2, 'modDate': "D:20260325192527+05'30'", 'total_pages': 14}, page_content='Finally, I accept the certifications indicated by the solar panel manufacturer for the \nability of the panels to withstand high wind and snow loads. Sincerely, \n \nXYZ \nStructural Engineer'),
 Document(metadata={'author': 'ruly', 'title': 'March 3rd, 2012', 'creationdate': '2026-03-25T19:25:27+05:30', 'trapped': '', 'keywords': '', 'subject': '', 'total_pages': 14, 'file_path': 'StructuralReport.pdf', 'producer': 'Microsoft® Word 2021', 'moddate': '2026-03-25T19:25:27+05:30', 'format': 'PDF 1.7', 'source':